# 1.3 RAG

In [1]:
# first run `uv pip install google-genai openai python-dotenv`
import os
import time
from google import genai
from google.genai.errors import APIError
from openai import OpenAI
from dotenv import load_dotenv

# Load your secret keys from the .env file
load_dotenv()

# ==============================================================================
# OPENAI (CHATGPT) MODEL CODES FOR API
# Note: The OpenAI API is technically Pay-As-You-Go (you buy credits), 
# but these are the models powering the free/standard tiers of the ChatGPT app:
#
# 1. "gpt-4o-mini"   -> The current default fast, cheap, and smart model.
#                       (Best equivalent to Gemini Flash. Highly recommended).
#
# 2. "gpt-4o"        -> The flagship, most intelligent model. Slower and costs more credits.
# 
# 3. "gpt-3.5-turbo" -> The older legacy model. (Now mostly replaced by gpt-4o-mini).
# ==============================================================================

# Initialize both AI clients securely using the environment variables
gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [2]:
def llm(prompt):
    """
    Main function to ask the AI a question. 
    It prioritizes Gemini, but falls back to ChatGPT if Gemini is overloaded.
    """
    
    # ---------------------------------------------------------
    # ROUTE 1: Try Gemini First
    # ---------------------------------------------------------
    try:
        print("Attempting to use Gemini...")
        
        # We call the new google-genai SDK method here
        response = gemini_client.models.generate_content(
            model='gemini-3.1-flash-lite', 
            contents=prompt
        )
        return response.text

    # ---------------------------------------------------------
    # ROUTE 2: Catch Gemini Server Errors and Fallback to ChatGPT
    # ---------------------------------------------------------
    except APIError as e:
        # Check if the error code means the server is just busy or rate-limited
        # 503 = Service Unavailable (Busy)
        # 429 = Too Many Requests (Rate limit hit)
        # 500 = Internal Server Error (Google is having a glitch)
        if e.code in [503, 429, 500]:
            print(f"Gemini unavailable (Error {e.code}). Falling back to ChatGPT...")
            # Route the exact same prompt to our backup function
            return ask_chatgpt(prompt)
        else:
            # If it's a different error (like an invalid API key), 
            # we want the program to crash so we can fix the bug.
            raise e
            
    except Exception as e:
        # Catch-all for basic internet connection drops or timeouts
        print(f"Gemini failed with a general error: {e}. Falling back to ChatGPT...")
        return ask_chatgpt(prompt)


def ask_chatgpt(prompt):
    """
    Helper function to handle the OpenAI (ChatGPT) API call.
    This only runs if Gemini fails.
    """
    try:
        # We use the standard OpenAI chat completions method here
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini", # <-- You can swap this with the models listed at the top!
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        # Extract the text from the OpenAI response structure
        return response.choices[0].message.content
        
    except Exception as e:
        # If BOTH Google and OpenAI servers are down, return a hard failure message
        return f"System Failure: Both Gemini and ChatGPT are unavailable. ChatGPT error: {e}"


# ==============================================================================
# TEST THE PIPELINE
# ==============================================================================
result = llm("Explain quantum computing in simple terms")

print("\n--- Final Output ---")
print(result)

Attempting to use Gemini...

--- Final Output ---
To understand quantum computing, first think about how a regular computer works.

### The Regular Computer (The Library)
A regular computer (like your phone or laptop) works using **bits**. Think of a bit like a light switch: it is either **On (1)** or **Off (0)**. 

Everything you do on a computer—watching videos, typing emails, playing games—is just a series of billions of these light switches flipping on and off. If you want to find a specific book in a library, a regular computer has to walk down every single aisle and check every single shelf, one by one, until it finds it. 

### The Quantum Computer (The Magic Librarian)
A quantum computer uses **qubits**. Thanks to the strange rules of quantum physics, a qubit can be **On, Off, or both at the same time.** This is called **superposition.**

Imagine if, instead of walking down every aisle, our "Magic Librarian" could exist in all the aisles simultaneously. They don't have to check 

In [3]:
prompt = input("What would you like to know?")
print("User's query: ", prompt)
print(llm(prompt))

User's query:  Tell me about LLM Zoomcamp.
Attempting to use Gemini...
**LLM Zoomcamp** is a popular, free, community-driven online bootcamp focused on teaching engineers how to build and deploy Large Language Model (LLM) applications. It is organized by **DataTalks.Club**, the same group behind the well-known "Data Engineering Zoomcamp" and "Machine Learning Zoomcamp."

Here is a breakdown of what you should know about it:

### 1. What is it about?
The course is designed to take you from a basic understanding of LLMs to building production-ready applications. It avoids being purely theoretical; instead, it focuses on the "engineering" side of AI—how to connect models to data, evaluate performance, and ship products.

### 2. Key Topics Covered
The curriculum usually covers the modern LLM tech stack, including:
*   **RAG (Retrieval-Augmented Generation):** This is the core of the course. You learn how to index documents, perform vector searches, and use context to get accurate answers f

# 1.4 Dataset

In [4]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [5]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)


1350

In [6]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

Each entry has:

- `id` - unique identifier
- `course` - course slug (e.g., machine-learning-zoomcamp)
- `section` - which section of the course
- `question` - the FAQ question
- `answer` - the FAQ answer

In the **RAG pipeline**, this dataset is our knowledge base:
1. We index all the documents (the search step)
2. When a student asks a question, we search the index
3. The search returns the most relevant FAQ entries
4. We give those entries to the LLM as context
5. The LLM generates an answer based on the context

The `question` and `answer` fields contain the text we'll search through. The `course` field lets us filter by course. For example, if a student asks about the data engineering course, we skip results from the ML course. The `section` field helps with ranking - knowing which part of the course a question belongs to is useful context.

**Data Preparation**: Our data is already prepared. However, in reality, data preparation is often the most time-consuming part of building a RAG system. You may need to scrape websites, parse PDFs, and clean and chunk documents. That work isn't visible here.We keep the focus on the GenAI side in this course. In your own projects, expect to spend real time on data preparation before you get to this point.

# 1.5 Search

In [7]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [8]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [9]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [10]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

In [11]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [12]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [13]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

# 1.6 Building the Prompt

In [14]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [15]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [16]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [17]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [19]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

# 1.7 The LLM


In [21]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [22]:
response.output[0]

ResponseOutputMessage(id='msg_02867771fd5974cf006a3a5bbafb848194be14c7012ccd4a7f', content=[ResponseOutputText(annotations=[], text='Yes — you can still join now.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [23]:
response.output[0].content[0].text

'Yes — you can still join now.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [24]:
response.output_text

'Yes — you can still join now.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

NameError: name 'ResponseUsage' is not defined